# Setup

In [1]:
# Setup
import json
import os
import sys
from datetime import datetime, timedelta
import time
from dotenv import load_dotenv, set_key
import random
from pathlib import Path

# hashing for signing
import hashlib
import hmac

# requests
import requests

# data munging
import pandas as pd
import numpy as np

# helper functions
from tiktok_api_helpers import *

## API Setup

In [2]:
new_access_token = False
use_refresh_token = False

In [3]:
if (new_access_token | use_refresh_token):

    # get new acces_token
    token_url = "https://auth.tiktok-shops.com/api/v2/token/get"

    if use_refresh_token:
        auth_code = os.environ.get("TIKTOK_REFRESH_TOKEN")
        
    params = {
        "app_key": app_key,
        "app_secret": app_secret,
        "auth_code": auth_code,
        "grant_type": "authorized_code",
    }
    
    response = requests.get(token_url, params=params, timeout=15)
    data = response.json()['data']
    access_token = data['access_token']
    print(data)

    # save tokens in .env file
    set_key(".env", "TIKTOK_ACCESS_TOKEN", data['access_token'])
    set_key(".env", "TIKTOK_REFRESH_TOKEN", data['refresh_token'])

else:
    access_token = os.environ.get("TIKTOK_ACCESS_TOKEN")
    refresh_token = os.environ.get("TIKTOK_REFRESH_TOKEN")

## Load Creator Data

In [4]:
# load all creators
df_creators_list = pd.read_excel("all_creators_sorted.xlsx", sheet_name="LIST_CREATOR", usecols=[1, 2, 25])
df_creators = pd.read_csv(CONSOLIDATED_CSV)

In [5]:
df_creators_all = df_creators_list \
    .merge(df_creators, how='left', on="username")

# Extract Top Creators by GMV and Units Sold

In [ ]:


DELAY_BETWEEN_PAGES = 5.0  # seconds between page requests

all_creators = []
search_key = ""
page_token = ""
page_num = 1

while True:
    result = search_creators_with_retry(
        gmv_ranges=["GMV_RANGE_10000_AND_ABOVE"],
        units_sold_ranges=["UNITS_SOLD_RANGE_100_1000", "UNITS_SOLD_RANGE_1000_AND_ABOVE"],
        search_key=search_key,
        page_token=page_token,
    )

    if result.get("code") != 0:
        print(f"  ⚠️  Page {page_num} failed, stopping here: {result}")
        break

    data = result.get("data", {}) or {}
    creators = data.get("creators", [])
    all_creators.extend(creators)

    print(f"Page {page_num}: {len(creators)} creator(s) (total so far: {len(all_creators)})")

    search_key = data.get("search_key", search_key)  # carry forward, per the doc's caching note
    page_token = data.get("next_page_token", "")
    if not page_token:
        break

    page_num += 1
    time.sleep(DELAY_BETWEEN_PAGES)

print(f"\nDone. {len(all_creators)} total creator(s) collected.")